# Fine-Tuning SegFormer for Improved Lane Detection 

---

- Conda env : [3dcv_playgrounds](../README.md#setup-a-conda-environment)

----

- Ref : https://learnopencv.com/segformer-fine-tuning-for-lane-detection/
- Data : http://128.32.162.150/bdd100k/video_parts/



In [19]:
!nvidia-smi

Fri Nov 14 20:28:50 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.08              Driver Version: 575.57.08      CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:01:00.0  On |                  N/A |
| 85%   70C    P2            104W /  250W |    8089MiB /  11264MiB |     70%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
import cv2
from PIL import Image
import torch
import numpy as np
from torchvision import transforms as TF
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation

In [4]:
# Load the trained model 
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SegformerForSemanticSegmentation.from_pretrained('nvidia/segformer-b2-finetuned-ade-512-512')

# Replace with the actual number of classes
model.config.num_labels = 2  

# Load the state from the fine-tuned model and set to model.eval() mode
model_path = "./temp_model/best_model.pth"
model.load_state_dict(torch.load(model_path))
model.to(device)
model.eval()


SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)

In [6]:
def test(model, input_video_path, output_video_path):
    # Video inference
    cap = cv2.VideoCapture(input_video_path)
    fourcc = cv2.VideoWriter_fourcc(*'XVID')
    out = cv2.VideoWriter(output_video_path, fourcc, 20.0, (int(cap.get(3)), int(cap.get(4))))

    # Perform transformations
    data_transforms = TF.Compose([
        TF.ToPILImage(),
        TF.Resize((360, 640)),
        TF.ToTensor(),
        TF.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Inference loop 
    while(cap.isOpened()):
        ret, frame = cap.read()
        if ret == True:
            # Preprocess the frame
            input_tensor = data_transforms(frame).unsqueeze(0).to(device)
            
            with torch.no_grad():
                outputs = model(pixel_values=input_tensor,return_dict=True)
                outputs = F.interpolate(outputs["logits"], size=(360, 640), mode="bilinear", align_corners=False)
                
                preds = torch.argmax(outputs, dim=1)
                preds = torch.unsqueeze(preds, dim=1)
                predicted_mask = (torch.sigmoid(preds) > 0.5).float()

            # Create an RGB version of the mask to overlay on the original frame
            mask_np = predicted_mask.cpu().squeeze().numpy()
            mask_resized = cv2.resize(mask_np, (frame.shape[1], frame.shape[0]))
            
            # Modify this section to create a green mask
            mask_rgb = np.zeros((mask_resized.shape[0], mask_resized.shape[1], 3), dtype=np.uint8)
            mask_rgb[:, :, 1] = (mask_resized * 255).astype(np.uint8)  # Set only the green channel

            # Post-processing for mask smoothening
            # Remove noise
            kernel = np.ones((3,3), np.uint8)
            opening = cv2.morphologyEx(mask_rgb, cv2.MORPH_OPEN, kernel, iterations=2)
            
            # Close small holes
            closing = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel, iterations=2)

            # Overlay the mask on the frame
            blended = cv2.addWeighted(frame, 0.65, closing, 0.6, 0)
            
            # Write the blended frame to the output video
            out.write(blended)
        else:
            break

    cap.release()
    out.release()
    cv2.destroyAllWindows()

In [8]:
import gdown
from pathlib import Path

id = "1IYIR4exGV4VxasqgVctaEmslxpXLCv_d"
test_video1_path = "./temp_data/BDD_test01.mp4"
gdown.download(id=id, output = test_video1_path)

Path("./temp_output").mkdir(exist_ok=True, parents=True)
output_path = "./temp_output/BDD_test01_detected.mp4"


test(model, test_video1_path, output_path)

Downloading...
From: https://drive.google.com/uc?id=1IYIR4exGV4VxasqgVctaEmslxpXLCv_d
To: /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/ADAS/Fine-Tuning-SegFormer-For-Lane-Detection/temp_data/BDD_test01.mp4
100%|██████████| 13.3M/13.3M [00:02<00:00, 5.92MB/s]
OpenCV: FFMPEG: tag 0x44495658/'XVID' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'


In [9]:
import gdown
from pathlib import Path

id = "1FEYQDFUACMnhWwRVNbCdKBFQLj0G8TMQ"
test_video2_path = "./temp_data/BDD_test02.mp4"

gdown.download(id=id, output = test_video2_path)

Path("./temp_output").mkdir(exist_ok=True, parents=True)
output_path = "./temp_output/BDD_test02_detected.mp4"


test(model, test_video2_path, output_path)

Downloading...
From: https://drive.google.com/uc?id=1FEYQDFUACMnhWwRVNbCdKBFQLj0G8TMQ
To: /home/hyunjae/110_HyunJae_Git/2025_Playgrounds/CV_Playgrounds/3DCV/ADAS/Fine-Tuning-SegFormer-For-Lane-Detection/temp_data/BDD_test02.mp4
100%|██████████| 9.89M/9.89M [00:01<00:00, 6.05MB/s]
OpenCV: FFMPEG: tag 0x44495658/'XVID' is not supported with codec id 12 and format 'mp4 / MP4 (MPEG-4 Part 14)'
OpenCV: FFMPEG: fallback to use tag 0x7634706d/'mp4v'
